# Generate Resampling Test Data from Tuples

Creates load profiles from explicit (energy, duration) tuples,
resamples them with the resampling method, and
saves the results as test cases.

In [16]:
import datetime
import os

from ethos_penalps.testing.load_profile.load_profil_creator import make_load_profile_entries_from_tuples
from ethos_penalps.testing.load_profile.resampling_helpers import (
    resample_load_profile,
    save_resampling_test_case,
)

## Output directory

In [17]:
CASES_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "cases")

## Case 1 — Equal-duration entries, uniform energy

Tests that uniform input is evenly distributed across resample bins.

In [18]:
CASE_NAME = "uniform_energy_equal_duration"

START = datetime.datetime(2021, 1, 1)
RESAMPLE_FREQUENCY = "1min"

tuples_uniform = [
    (100.0, datetime.timedelta(minutes=2)),
    (100.0, datetime.timedelta(minutes=2)),
    (100.0, datetime.timedelta(minutes=2)),
    (100.0, datetime.timedelta(minutes=2)),
    (100.0, datetime.timedelta(minutes=2)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_uniform)
end = START + sum((d for _, d in tuples_uniform), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 5, Bins: 10
Total energy in:  500.000000 MJ
Total energy out: 500.000000 MJ
Saved case 'uniform_energy_equal_duration' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/uniform_energy_equal_duration/
  input_load_profile.csv                   0.7 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   1.3 KB
  resampled_vectorized.csv                 1.3 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/uniform_energy_equal_duration'

## Case 2 — Mixed durations, varying energy

Tests resampling when entries have different lengths and energy values.

In [19]:
CASE_NAME = "mixed_duration_varying_energy"

tuples_mixed = [
    (500.0, datetime.timedelta(seconds=30)),
    (150.0, datetime.timedelta(minutes=1, seconds=30)),
    (200.0, datetime.timedelta(minutes=2)),
    (150.0, datetime.timedelta(minutes=1, seconds=30)),
    (27.0, datetime.timedelta(seconds=30)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_mixed)
end = START + sum((d for _, d in tuples_mixed), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 5, Bins: 6
Total energy in:  1027.000000 MJ
Total energy out: 1027.000000 MJ
Saved case 'mixed_duration_varying_energy' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/mixed_duration_varying_energy/
  input_load_profile.csv                   0.7 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.8 KB
  resampled_vectorized.csv                 0.8 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/mixed_duration_varying_energy'

## Case 3 — Single long entry

Tests that a single entry spanning the full range is evenly split across bins.

In [20]:
CASE_NAME = "single_long_entry"

tuples_single = [
    (5000.0, datetime.timedelta(minutes=10)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_single)
end = START + sum((d for _, d in tuples_single), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 1, Bins: 10
Total energy in:  5000.000000 MJ
Total energy out: 5000.000000 MJ
Saved case 'single_long_entry' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/single_long_entry/
  input_load_profile.csv                   0.2 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   1.3 KB
  resampled_vectorized.csv                 1.3 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/single_long_entry'

## Case 4 — Sub-minute entries with 5min resampling

Tests aggregation when many short entries are merged into fewer, longer bins.

In [21]:
CASE_NAME = "sub_minute_entries_5min_resample"

RESAMPLE_FREQUENCY_5MIN = "5min"

tuples_sub_minute = [
    (10.0, datetime.timedelta(seconds=15)),
    (20.0, datetime.timedelta(seconds=45)),
    (30.0, datetime.timedelta(seconds=20)),
    (40.0, datetime.timedelta(seconds=10)),
    (50.0, datetime.timedelta(seconds=30)),
    (60.0, datetime.timedelta(seconds=60)),
    (70.0, datetime.timedelta(seconds=25)),
    (80.0, datetime.timedelta(seconds=55)),
    (90.0, datetime.timedelta(seconds=40)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_sub_minute)
end = START + sum((d for _, d in tuples_sub_minute), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY_5MIN)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY_5MIN},
    case_name=CASE_NAME,
)

Entries: 9, Bins: 1
Total energy in:  450.000000 MJ
Total energy out: 450.000000 MJ
Saved case 'sub_minute_entries_5min_resample' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/sub_minute_entries_5min_resample/
  input_load_profile.csv                   1.1 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.2 KB
  resampled_vectorized.csv                 0.2 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/sub_minute_entries_5min_resample'

## Case 5 — Short spike not aligned to grid

A 20s high-energy spike sits entirely within a single 1min bin, not touching any grid boundary.
Tests that energy is correctly assigned when an entry doesn't cross a resample boundary.

In [22]:
CASE_NAME = "short_spike_not_grid_aligned"

RESAMPLE_FREQUENCY = "1min"

# 40s idle, 20s spike at high energy, 120s idle -> total 3min
tuples_spike = [
    (0.0, datetime.timedelta(seconds=40)),
    (9000.0, datetime.timedelta(seconds=20)),
    (0.0, datetime.timedelta(seconds=120)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_spike)
end = START + sum((d for _, d in tuples_spike), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 3, Bins: 3
Total energy in:  9000.000000 MJ
Total energy out: 9000.000000 MJ
Saved case 'short_spike_not_grid_aligned' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/short_spike_not_grid_aligned/
  input_load_profile.csv                   0.4 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.4 KB
  resampled_vectorized.csv                 0.4 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/short_spike_not_grid_aligned'

## Case 6 — Zero energy entries

Entries with zero energy interspersed with non-zero ones.
Tests that zeros propagate correctly and don't cause division issues.

In [23]:
CASE_NAME = "zero_energy_entries"

RESAMPLE_FREQUENCY = "1min"

tuples_zero = [
    (0.0, datetime.timedelta(minutes=1)),
    (500.0, datetime.timedelta(minutes=1)),
    (0.0, datetime.timedelta(minutes=1)),
    (0.0, datetime.timedelta(minutes=1)),
    (300.0, datetime.timedelta(minutes=1)),
    (0.0, datetime.timedelta(minutes=1)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_zero)
end = START + sum((d for _, d in tuples_zero), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 6, Bins: 6
Total energy in:  800.000000 MJ
Total energy out: 800.000000 MJ
Saved case 'zero_energy_entries' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/zero_energy_entries/
  input_load_profile.csv                   0.7 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.8 KB
  resampled_vectorized.csv                 0.8 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/zero_energy_entries'

## Case 7 — Entries exactly aligned to grid

Entries are exactly 1min long and perfectly aligned to the 1min resample grid.
Tests the boundary-inclusive/exclusive logic when entry edges coincide with bin edges.

In [24]:
CASE_NAME = "entries_aligned_to_grid"

RESAMPLE_FREQUENCY = "1min"

tuples_aligned = [
    (100.0, datetime.timedelta(minutes=1)),
    (200.0, datetime.timedelta(minutes=1)),
    (300.0, datetime.timedelta(minutes=1)),
    (400.0, datetime.timedelta(minutes=1)),
    (500.0, datetime.timedelta(minutes=1)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_aligned)
end = START + sum((d for _, d in tuples_aligned), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 5, Bins: 5
Total energy in:  1500.000000 MJ
Total energy out: 1500.000000 MJ
Saved case 'entries_aligned_to_grid' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/entries_aligned_to_grid/
  input_load_profile.csv                   0.7 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.7 KB
  resampled_vectorized.csv                 0.7 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/entries_aligned_to_grid'

## Case 8 — One large spike, rest tiny

One entry contains 99% of total energy in a very short duration.
Tests that extreme power values are distributed correctly across bins.

In [25]:
CASE_NAME = "large_spike_rest_tiny"

RESAMPLE_FREQUENCY = "1min"

tuples_spike_dominant = [
    (1.0, datetime.timedelta(minutes=2)),
    (1.0, datetime.timedelta(seconds=20)),
    (10000.0, datetime.timedelta(seconds=10)),
    (1.0, datetime.timedelta(seconds=30)),
    (1.0, datetime.timedelta(minutes=2)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_spike_dominant)
end = START + sum((d for _, d in tuples_spike_dominant), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 5, Bins: 5
Total energy in:  10004.000000 MJ
Total energy out: 10004.000000 MJ
Saved case 'large_spike_rest_tiny' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/large_spike_rest_tiny/
  input_load_profile.csv                   0.7 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.7 KB
  resampled_vectorized.csv                 0.7 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/large_spike_rest_tiny'

## Case 9 — Many entries within a single bin

50 short entries all fit within a single 5min resample bin.
Tests aggregation of many small entries into one output bin.

In [26]:
CASE_NAME = "many_entries_single_bin"

RESAMPLE_FREQUENCY_5MIN = "5min"

# 50 entries of 6s each = 300s = 5min, all in one bin
tuples_many = [(float(i + 1), datetime.timedelta(seconds=6)) for i in range(50)]

entries = make_load_profile_entries_from_tuples(START, tuples_many)
end = START + sum((d for _, d in tuples_many), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY_5MIN)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY_5MIN},
    case_name=CASE_NAME,
)

Entries: 50, Bins: 1
Total energy in:  1275.000000 MJ
Total energy out: 1275.000000 MJ
Saved case 'many_entries_single_bin' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/many_entries_single_bin/
  input_load_profile.csv                   5.8 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   0.2 KB
  resampled_vectorized.csv                 0.2 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/many_entries_single_bin'

## Case 10 — One entry spanning many bins

A short entry followed by one long entry spanning 10+ resample bins.
Tests the split logic when a single entry must be distributed across many output bins.

In [27]:
CASE_NAME = "one_entry_spanning_many_bins"

RESAMPLE_FREQUENCY = "1min"

tuples_long_span = [
    (50.0, datetime.timedelta(seconds=30)),
    (100.0, datetime.timedelta(seconds=30)),
    (6000.0, datetime.timedelta(minutes=12)),
    (50.0, datetime.timedelta(seconds=30)),
    (100.0, datetime.timedelta(seconds=30)),
]

entries = make_load_profile_entries_from_tuples(START, tuples_long_span)
end = START + sum((d for _, d in tuples_long_span), datetime.timedelta())

meta, result = resample_load_profile(entries, START, end, RESAMPLE_FREQUENCY)
print(f"Entries: {len(entries)}, Bins: {len(result.list_of_load_profiles)}")
print(f"Total energy in:  {meta.total_energy:.6f} {meta.energy_unit}")
print(f"Total energy out: {result.total_energy:.6f} {result.energy_unit}")

save_resampling_test_case(
    meta=meta,
    result=result,
    cases_dir=CASES_DIR,
    parameters={"start": START.isoformat(), "end": end.isoformat(), "resample_frequency": RESAMPLE_FREQUENCY},
    case_name=CASE_NAME,
)

Entries: 5, Bins: 14
Total energy in:  6300.000000 MJ
Total energy out: 6300.000000 MJ
Saved case 'one_entry_spanning_many_bins' to /fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/one_entry_spanning_many_bins/
  input_load_profile.csv                   0.7 KB
  parameters.json                          0.1 KB
  resampled_original.csv                   1.8 KB
  resampled_vectorized.csv                 1.8 KB


'/fast/home/j-belina/ethos_penalps/test/generate/load_profile/aligned/cases/one_entry_spanning_many_bins'